### Exceptions

When an error occurs, what can we do?
```
Print an error message
Stop the execution of a program
Indicate the error by returning a special value, like -1 or None
Ignore the error
...
```

These solutions tend to combine the indication of a problem and the reaction to the problem indication. The behaviour of the program in error situations cannot be changed, they are fixed in the implementation of the function. When an erroneous situation is noticed, it may not be clear how to handle the situation. Usually the user or an instance that called a function knows what to do.

## Good Instinct to Question This — Let's Trace the Full Context

You're right to push back, but I think this sentence is describing a **specific, narrower** situation than "error handling in general" — it's contrasting **two particular designs**, and the sentence only makes sense once you see what it's being compared against.

---

### The Likely Setup — "Return Special Value" vs "Raise Exception"

This paragraph is almost certainly introducing **why exceptions exist**, by first describing the **older/simpler alternative**: functions that signal errors by **returning a special value** (like `-1`, `None`, or `False`) instead of raising an exception.

```python
def find_index(lst, item):
    """Returns the index, or -1 if not found."""
    for i, x in enumerate(lst):
        if x == item:
            return i
    return -1     # "error indication" baked directly into the return value
```

**This** is the design the sentence is criticizing — and here, *"behaviour... cannot be changed"* means something quite specific: **the function itself decides, once and for all, that "not found" = `-1`.** Every single caller is stuck with that exact convention — they can't ask the function to instead print a warning, or crash loudly, or return `None` — the `-1` is **hardcoded inside the function's own logic**.

```python
idx = find_index([1,2,3], 99)
if idx == -1:               # EVERY caller must remember this exact convention
    print("not found")
```

That's what *"fixed in the implementation of the function"* means — the **reaction** (what the -1 even means, how you're supposed to notice it) is welded into the function's source code, not something the caller can influence.

---

### Where Your Objection Is Right — Exceptions DO Let You Change the Reaction

Compare with the exception-based design, which this text is very likely building toward:

```python
def find_index(lst, item):
    for i, x in enumerate(lst):
        if x == item:
            return i
    raise ValueError(f"{item} not found")
```

```python
try:
    idx = find_index([1,2,3], 99)
except ValueError:
    print("Custom reaction A")     # caller #1 chooses this
```

```python
try:
    idx = find_index([1,2,3], 99)
except ValueError:
    idx = -1                          # caller #2 chooses a totally different reaction!
```

**Here**, the function only handles the **detection** (raising `ValueError`) — the **reaction** is left entirely to whoever calls it, and different callers can react completely differently to the exact same error. This is precisely your point: *"we want to react differently, without breaking the system"* — and exceptions are the mechanism that achieves that flexibility.

---

### So — Is the Sentence Correct?

**Yes, but only for the specific (older, more limited) design being described in that sentence** — not as a universal claim about all error-handling in Python.

> *"These solutions tend to combine the indication of a problem and the reaction to the problem"*

This is the **diagnosis**: return-value-based error signaling **welds together** two things that should be separate — *"something went wrong"* and *"here's what to do about it."*

> *"The behaviour of the program in error situations cannot be changed"*

This means: **with THIS particular design** (special return values), the reaction is baked in — the caller has no say in it, beyond checking the special value manually every time.

The paragraph is very likely setting up a **contrast** — "here's the old, rigid way (return values) → here's why exceptions are better (separates detection from reaction, letting the caller decide)." Exceptions are almost certainly introduced in the very next section, as the solution to exactly the limitation you're correctly identifying.

---

### The Corrected, Precise Reading

> The sentence is **narrowly true**, describing return-value-based error signaling specifically — where "problem happened" and "what -1/None/False means" are permanently fused into the function's own code, leaving the caller with no ability to choose a different reaction beyond manually checking that fixed value. It is **not** a claim that Python's error handling *in general* can't be customized — exceptions (the very next topic, almost certainly) exist **specifically** to fix this limitation, by letting each caller decide its own reaction via `try`/`except`, exactly as you intuited.

Your reasoning was sound — the confusion was just that the sentence describes the **problem being set up**, not the eventual solution. Once exceptions get introduced (if they haven't in your next section), this whole tension resolves. 🎯

Most modern computer languages have a system called ```exception handling```. This system separates the recognition of errors and the handling of these situations. We can signal an error or anomalous situation by raising an exception. Exceptions can be raised in Python with the raise statement:

* raise instance
* raise exception class [, expression]

In the second form, if the expression exists, it is a tuple of parameters given to exception class.

The functions of the Python standard library raise exceptions in error situations. Sometimes exceptions aren’t really errors. For example, when an iterator runs out of elements, it will signal this by raising the StopIteration exception. Another less erroneus exception is the Warning exceptio

## Now It's Fully Clear — Your Original Reading Was Actually Correct

Thank you for the full context — this changes things. Let me re-examine, because I think I **over-corrected** in my earlier answer. Reading the whole note together, your instinct at the very start was right, and I owe you a more careful re-trace.

---

### The Note's Actual Structure

The paragraph you quoted describes **ONE specific approach to error handling** — the bulleted list at the top:

```
* Print an error message
* Stop the execution of a program
* Indicate the error by returning a special value, like -1 or None
* Ignore the error
```

**This whole list is what "these solutions" refers to.** And crucially — look closely — this list **already includes things like "stop the program" and "print a message"**, not just "return -1." So "these solutions" is broader than I gave it credit for in my earlier answer — it's describing **any approach where the function itself decides the reaction**, not narrowly just "return-value signaling."

---

### Re-Reading "Cannot Be Changed" With This Broader List in Mind

> *"These solutions tend to combine the indication of a problem and the reaction to the problem indication. The behaviour... cannot be changed, they are fixed in the implementation of the function."*

Now it makes more sense as a **general critique of the pre-exception approach as a whole**: if a function is **written** to, say, print an error and stop the program the moment something goes wrong — the **caller has no way to intervene**. The function's code **directly performs** the reaction (printing, stopping, returning -1) — there's no hook, no separation, no opportunity for the caller to say *"actually, I'd rather handle this differently."*

```python
def risky_old_style(x):
    if x < 0:
        print("Error: negative value!")     # reaction is BAKED IN — happens unconditionally
        return -1                              # caller can't prevent this print, or choose differently
    return x * 2
```

Whoever calls `risky_old_style(-5)` is **stuck** with "a message gets printed, -1 comes back" — full stop. No `try`/`except`-style opt-out exists in this design.

---

### Why Exceptions Are Then Introduced as THE Fix

> *"Most modern languages have exception handling. This system **separates** the recognition of errors and the handling..."*

This is the pivot sentence — and it directly validates your original point. The **entire reason** exceptions are being introduced, right after that "cannot be changed" claim, is **precisely** to solve the rigidity being described. The note is structured as:

```
Problem:   "old-style reactions are baked into the function, caller is stuck"
Solution:  "exceptions separate detection from handling, giving the caller control"
```

So your original objection — *"the behavior changes because we want to react differently, without breaking the system"* — isn't just compatible with the note; it's **exactly the point the note goes on to make**, one sentence later. You correctly anticipated the note's own argument before reading it.

---

### Where I Went Slightly Wrong Earlier

In my first answer, I said the "cannot be changed" sentence was **narrowly** about return-value signaling specifically. Having now seen the full bulleted list, I was too narrow — it's describing the **whole family** of function-decides-the-reaction approaches (print, stop, return special value, ignore) as a **group**, contrasted **as a group** against exceptions. My correction stands in spirit (exceptions do restore caller flexibility) but I mis-scoped exactly what "these solutions" covered.

---

### The Full Corrected Picture

```
OLD APPROACH (the bulleted list):
   function decides: "if error, PRINT + STOP + return -1"
   → reaction is WELDED into the function
   → every caller gets the SAME behavior, no matter what
   → "cannot be changed" = TRUE, for this specific design

EXCEPTIONS (the fix, introduced right after):
   function decides: "if error, RAISE SomeException"
   → function ONLY signals — does not decide what happens next
   → caller CHOOSES: try/except with whatever reaction it wants
   → "cannot be changed" claim NO LONGER APPLIES — that's the whole upgrade
```

---

### Now, Placing `StopIteration` and `Warning` Correctly Within This Story

These aren't a separate topic — they're the note's way of showing that exceptions are used for **more than fixing the old problem-list**. Since exceptions are now a **general-purpose signal**, Python reuses the same tool even for things that were **never** on that original error-reaction list at all:

- `StopIteration` — signals *"iteration finished"* — not an error, never would have appeared in a "what do we do about errors" list to begin with
- `Warning` — signals *"heads up, non-fatal"* — softer than any of the original bullet points

The note is subtly showing: once you have a **general "interrupt and signal" mechanism** (exceptions), it turns out to be useful for **all sorts of non-error signaling too** — not just the error-reaction problem it was originally introduced to fix.

---

### The Honest Bottom Line

> Your very first instinct was right: the note is building toward "reactions CAN differ, without breaking the system" — that's the **entire selling point** of exceptions, stated explicitly as *"separates the recognition of errors and the handling of these situations."* The "cannot be changed" sentence describes the **problem exceptions were invented to solve**, not a permanent limitation of Python. I initially scoped that sentence too narrowly to just return-values; with the full note in front of us, it's clearly describing the whole pre-exception bulleted list as a group. No contradiction anywhere in the note itself — and none in my two answers taken together, though my first one undersold how directly your objection was already the note's own thesis. 🎯

In [ ]:
try:
    # here are the statements that can cause exceptions
except (Exceptionname1, Exceptionname2, ...):
    # here we handle the exceptions
else:
    # this gets executed if try-block caused no exceptions
finally:
    # this is always executed, clean-up code


In [ ]:
L=[1,2,3]

try:
    print(L[3])
except IndexError:
    print("Index does not exit")

In [7]:
def compute_average(L):
    n=len(L)
    s=sum(L)
    return float(s)/n

mylist=[]

while True:
    try:
        x=float(input("Give a number (non-number quites): "))
        mylist.append(x)
    except ValueError:
        break

try:
    average=compute_average(mylist)
    print(f"Average is {average}")
except ZeroDivisionError:
    # and the error is handled here
    if len(mylist)==0:
        print("Tried to compute the average of empty list of numbers")
    else:
        print("Something strange happened")

Tried to compute the average of empty list of numbers


## Tracing What Happens on Non-Numeric Input

Let's walk through the code step by step, focusing on the `try`/`except` inside the `while` loop.

---

### The Relevant Block

```python
while True:
    try:
        x = float(input("Give a number (non-number quites): "))
        mylist.append(x)
    except ValueError:
        break
```

---

### Step-by-Step — User Types a Character (e.g., `"q"`)

**1. `input(...)`** — reads whatever the user typed, as a **string**:
```python
input("Give a number...")     # user types "q" → returns "q"
```

**2. `float("q")`** — attempts to convert that string to a number:
```python
float("q")
# ✗ ValueError: could not convert string to float: 'q'
```

This is the **exact same error** you saw way back with `float('')` on blank lines, and `float("abc")` in your earlier `summary()` exercise — non-numeric text simply cannot become a `float`.

**3. The exception fires** — and since we're inside a `try:` block, Python **doesn't crash**. Instead, it immediately jumps to the matching `except`:

```python
except ValueError:
    break
```

**4. `break` runs** — this **exits the `while True:` loop entirely**. The loop stops asking for more numbers.

---

### The Line `mylist.append(x)` NEVER RUNS

This is worth highlighting specifically — because the exception happens **inside** the `try` block, at the `float(...)` call, execution **jumps straight to `except`** the instant the error occurs. The very next line, `mylist.append(x)`, is **skipped entirely** — `x` was never even successfully assigned a value, so there's nothing valid to append anyway.

```
try:
    x = float(input(...))   # ← ERROR HAPPENS HERE, on non-numeric input
    mylist.append(x)          # ← this line is SKIPPED — never reached!
except ValueError:
    break                       # ← control jumps straight here
```

---

### What Happens AFTER `break`

The loop ends, and execution continues with whatever comes after the `while` loop — the second `try` block:

```python
try:
    average = compute_average(mylist)
    print(f"Average is {average}")
except ZeroDivisionError:
    if len(mylist) == 0:
        print("Tried to compute the average of empty list of numbers")
    else:
        print("Something strange happened")
```

**Two possible outcomes here, depending on WHEN the user typed a character:**

---

### Case A — User Types a Character IMMEDIATELY (first input)

```
mylist = []       ← still empty, nothing was ever appended
```

```python
compute_average([])
```

Inside `compute_average`:
```python
n = len(L)        # → 0
s = sum(L)          # → 0
return float(s)/n     # → 0.0 / 0     ✗ ZeroDivisionError!
```

This propagates up and is caught by the **outer** `except ZeroDivisionError:` — and since `len(mylist) == 0`, you'd see:

```
Tried to compute the average of empty list of numbers
```

---

### Case B — User Enters Some Numbers FIRST, Then a Character

```
User types: 5, 10, 15, then "q"
mylist = [5.0, 10.0, 15.0]     ← numbers collected BEFORE the character stopped the loop
```

```python
compute_average([5.0, 10.0, 15.0])
# → n=3, s=30.0 → 30.0/3 = 10.0
```

No error this time — the `except ZeroDivisionError` block never triggers, and you'd see:

```
Average is 10.0
```

---

### The Full Flow, Visualized

```
Loop asks for numbers, one at a time:
    "5"    → float("5")=5.0    → appended    → loop continues
    "10"   → float("10")=10.0   → appended    → loop continues
    "q"     → float("q")  ✗ ValueError!        → break → LOOP EXITS

mylist is now whatever was collected BEFORE the "q" (could be empty, could have items)

compute_average(mylist) runs:
    if mylist was empty  → ZeroDivisionError → caught → "empty list" message
    if mylist had items    → normal average    → printed successfully
```

---

### The One-Sentence Summary

> Typing a non-numeric character triggers `ValueError` inside `float(...)`, which is immediately caught by the inner `except ValueError: break` — this **skips** the `.append(x)` line entirely and **exits the loop**. What happens next depends entirely on whether any *valid* numbers were entered before that point: an empty `mylist` triggers a `ZeroDivisionError` inside `compute_average` (caught by the outer `try`, printing the "empty list" message), while a non-empty `mylist` computes and prints a normal average. 🎯